# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [124]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [125]:
# TODO: set n_missing_study to the number of blank study_hours_reported values
n_missing_study = students["study_hours_reported"].isna().sum()

n_missing_study

np.int64(255)

**Your justification (1–2 sentences):** I believe the best approach to clean the data related to the missing value in the column `study_hours_reported` is to utilize the method titled as **Use a global constant to fill in the missing value** and specifically would use the text representation of `Null` as "Null". In this case "Null" will represent we simply do not have that value and covers the cases of failure to enter the data, failure to collect the data, and when it would be innappropriate to collect those data.

In [126]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [127]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups
students["housing_clean"] = students["housing"].str.strip().str.lower().map({
    "on-campus": "on-campus",
    "off-campus": "off-campus",
    "with family": "with family"
    })

In [128]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'off-campus': 832, 'on-campus': 590, 'with family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [129]:
# TODO
impossible_ages = sorted(students.loc[(students["age"] > 100) | (students["age"] < 10), "age"].unique())
n_neg_work = students[students["work_hours_per_week"] < 0]["work_hours_per_week"].count()

In [130]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [131]:
# TODO: build students_dedup (one row per student)
students_dedup = students.drop_duplicates(subset = ["student_id"])

**Why not de-dupe enroll / activity? (1 sentence):** `enroll` and `activity` both are tables that have composite primary keys. For `activity` it is a combination of studnet_id and week. For `enroll` it is a combination of sid, course, and term.

In [132]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [133]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
analysis = students_dedup.copy()
analysis["standardized_numeric_key"] = analysis["student_id"].str.replace("NU-", "").astype(int)
analysis = analysis.merge(activity.groupby("student_id")["minutes_active"].sum(), on = "student_id", how = "left")
analysis = analysis.rename(columns = {"minutes_active": "total_minutes_active"})



In [134]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [135]:
# TODO
matched = enroll["sid"].isin(set(analysis["standardized_numeric_key"])).sum()

In [136]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [137]:
# TODO
print(students["enrollment_date"].isna().sum())
dates_parsed = pd.to_datetime(students["enrollment_date"], format = "mixed", errors = "coerce")
print(dates_parsed.isna().sum())

0
0


In [138]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [139]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]
analysis["study_z"] = (analysis["study_hours_reported"] - analysis["study_hours_reported"].mean()) / analysis["study_hours_reported"].std()
analysis["gpa_band"] = pd.cut(analysis["final_gpa"], bins = [-0.01, 1, 2, 3, 4])

In [140]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [141]:
# TODO: write analysis to northgate_clean.csv
analysis.to_csv("northgate_clean.csv", index = False)

In [142]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- _decision → justification_
- _missing values_ → I elected to utilize the method of labeling missing data with a standardized phrase as I believe it is the best method to acknowledge the missing data as truly missing. By doing something like replacing it with a chosen value the risk of altering reporting is high.
- _housing_ → Housing categorical values were a mess. By standardizing the way the three primary categories were in the table, it is possible to enhance the reporting any time we need to count or group by those categorical values.
- _impossible values_ → Impossible values are clearly bad data and especially in the case of age are outliers that would both potentially skew data and just make us look unprofessional if left in reports such as when giving a range.
- _duplicates_ → Duplicates are also an example of bad data entry practices and ultimately bad index management. We needed to deduplicate the data to avoid duplicated data having a dispoportionate impact on reports and ensure we can justify the data we are providing to stakeholders.
- _key_ → By making a standardized numerical key, we worked around the issue of having inconsistent keys in the database. They clearly are relational keys, but because they were stored in two different ways it was not possible to merge those data in a meaningful way without making a standardized key.
- _dates_ → Having dates in multiple formats in the same column creates inconsistency and makes it far more difficult to perform meaningful operations on the column. We standardized the format of the dates (in a discrete variable) so that it is possible to utilize the date values for reporting.

### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [143]:
# TODO: compute the mean GPA and explore one relationship on the CLEAN data
overall_GPA_mean = analysis["final_gpa"].mean()
# print(f"{overall_GPA_mean:.2f}")

housing_GPA_means = analysis.groupby("housing_clean", as_index = False)["final_gpa"].mean()
housing_GPA_means.columns = ["Housing Type", "Average GPA"]
pd.set_option("display.float_format", "{:.2f}".format)
print(housing_GPA_means.to_string(index = False))

Housing Type  Average GPA
  off-campus         2.33
   on-campus         2.28
 with family         2.29


Cleaning change the ability to conduct this report by creating standard categories for the different housing types stored in the student_records table. Prior to this, it would have been difficult to group by the labels so that the average would be accurate for those groups. Of note, the small report above begs the question if there's correlation in the number of hours students have earned and in the rates of selecting off-campus housing as that may be a justification for the disparity between off-campus and the other two housing types.

In [144]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
